In [5]:
import os
from dotenv import load_dotenv
from openai import OpenAI

In [6]:
load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

# Check the key

if not api_key:
    print("No API key was found - please head over to the troubleshooting notebook in this folder to identify & fix!")
elif not api_key.startswith("sk-proj-"):
    print("An API key was found, but it doesn't start sk-proj-; please check you're using the right key - see troubleshooting notebook")
elif api_key.strip() != api_key:
    print("An API key was found, but it looks like it might have space or tab characters at the start or end - please remove them - see troubleshooting notebook")
else:
    print("API key found and looks good so far!")

API key found and looks good so far!


In [7]:
message = "Hello, GPT! This is my first ever message to you! Hi!"

messages = [{"role": "user", "content": message}]

messages

[{'role': 'user',
  'content': 'Hello, GPT! This is my first ever message to you! Hi!'}]

## Chat completions API

It's known as a Python Client Library.

It's nothing more than a wrapper around making this exact call to the http endpoint.

In [8]:
openai = OpenAI()

response = openai.chat.completions.create(model="gpt-5-nano", messages=messages)
response.model_dump_json()

'{"id":"chatcmpl-EJrthKx4nLoAf3ugLpvvojDHkLwXL","choices":[{"finish_reason":"stop","index":0,"logprobs":null,"message":{"content":"Hi there! Welcome — nice to meet you.\\n\\nI can help with a lot of things: explain concepts, answer questions, help with writing (emails, essays, resumes), coding and debugging, math problems, planning and brainstorming, translations, and much more.\\n\\nWhat would you like to do today? If you’re not sure, here are quick starters:\\n- Explain something in simple terms (e.g., photosynthesis)\\n- Draft a short email or message\\n- Help with a writing or homework task\\n- Start learning something new (a bit of Python, a math topic, etc.)\\n- Plan something (a trip, a meal plan, a study schedule)\\n\\nFeel free to tell me your name, and what you’re curious about or working on. How can I assist you first?","refusal":null,"role":"assistant","annotations":[],"audio":null,"function_call":null,"tool_calls":null}}],"created":1788404121,"model":"gpt-5-nano-2025-08-07

In [9]:
response.choices[0].message.content

'Hi there! Welcome — nice to meet you.\n\nI can help with a lot of things: explain concepts, answer questions, help with writing (emails, essays, resumes), coding and debugging, math problems, planning and brainstorming, translations, and much more.\n\nWhat would you like to do today? If you’re not sure, here are quick starters:\n- Explain something in simple terms (e.g., photosynthesis)\n- Draft a short email or message\n- Help with a writing or homework task\n- Start learning something new (a bit of Python, a math topic, etc.)\n- Plan something (a trip, a meal plan, a study schedule)\n\nFeel free to tell me your name, and what you’re curious about or working on. How can I assist you first?'

## Another way of calling LLM via direct endpoint

In [19]:
import requests

headers = {"Authorization": f"Bearer {api_key}", "Content-Type": "application/json"}

payload = {
    "model": "gpt-5-nano",
    "messages": [
        {"role": "user", "content": "Tell me a joke about Dentist."}]
}

payload

{'model': 'gpt-5-nano',
 'messages': [{'role': 'user', 'content': 'Tell me a joke about Dentist.'}]}

In [20]:
response = requests.post(
    "https://api.openai.com/v1/chat/completions",
    headers=headers,
    json=payload
)

response.json()

{'id': 'chatcmpl-EJrw1WQqWMFVVZSs0f8qUdznNuxri',
 'object': 'chat.completion',
 'created': 1788404265,
 'model': 'gpt-5-nano-2025-08-07',
 'choices': [{'index': 0,
   'message': {'role': 'assistant',
    'content': "Here's a quick one: Why did the dentist become a baseball coach? Because he knows the drill.\n\nWant another?",
    'refusal': None,
    'annotations': []},
   'finish_reason': 'stop'}],
 'usage': {'prompt_tokens': 13,
  'completion_tokens': 608,
  'total_tokens': 621,
  'prompt_tokens_details': {'cached_tokens': 0, 'audio_tokens': 0},
  'completion_tokens_details': {'reasoning_tokens': 576,
   'audio_tokens': 0,
   'accepted_prediction_tokens': 0,
   'rejected_prediction_tokens': 0}},
 'service_tier': 'default',
 'system_fingerprint': None}

In [21]:
response.json()["choices"][0]["message"]["content"]

"Here's a quick one: Why did the dentist become a baseball coach? Because he knows the drill.\n\nWant another?"

## Types of roles in messages

- **A system prompt** that tells them what task they are performing and what tone they should use
- **A user prompt** -- the conversation starter that they should reply to

In [13]:
messages = [
    {"role": "system", "content": "You are a Ricky Gervais."},
    {"role": "user", "content": "Tell me a new original joke."}
    ]

openai = OpenAI()

response = openai.chat.completions.create(model="gpt-5-nano", messages=messages)
response.choices[0].message.content

'Here\'s a fresh one in my style: I told my therapist I feel invisible. He handed me a mirror and said, "There you are."'

# What is the openai package?

It's known as a Python Client Library.

It's nothing more than a wrapper around making this exact call to the http endpoint.

It just allows you to work with nice Python code instead of messing around with janky json objects.

But that's it. It's open-source and lightweight. Some people think it contains OpenAI model code - it doesn't!

## And then this great thing happened:

OpenAI's Chat Completions API was so popular, that the other model providers created endpoints that are identical.

They are known as the "OpenAI Compatible Endpoints".

For example, google made one here: https://generativelanguage.googleapis.com/v1beta/openai/

And OpenAI decided to be kind: they said, hey, you can just use the same client library that we made for GPT. We'll allow you to specify a different endpoint URL and a different key, to use another provider.

So you can use:

```python
gemini = OpenAI(base_url="https://generativelanguage.googleapis.com/v1beta/openai/", api_key="AIz....")
gemini.chat.completions.create(...)
```

And to be clear - even though OpenAI is in the code, we're only using this lightweight python client library to call the endpoint - there's no OpenAI model involved here.

In [22]:
GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"

load_dotenv(override=True)

google_api_key = os.getenv("GOOGLE_API_KEY")

if not google_api_key:
    print("No API key was found - please be sure to add your key to the .env file, and save the file! Or you can skip the next 2 cells if you don't want to use Gemini")
elif not google_api_key.startswith(("AIz", "AQ.")):
    print("An API key was found, but it doesn't start with AIz or AQ.")
else:
    print("API key found and looks good so far!")

API key found and looks good so far!


In [23]:
gemini = OpenAI(base_url=GEMINI_BASE_URL, api_key=google_api_key)

response = gemini.chat.completions.create(model="gemini-3.1-flash-lite", messages=[{"role": "user", "content": "Tell me a fun fact"}])

response.choices[0].message.content

'Did you know that **sea otters hold hands when they sleep**? \n\nThey do this to keep from drifting apart in the water while they snooze. Sometimes, they will even anchor themselves to a piece of giant kelp to stay extra secure! These groups of resting otters are known as a "raft."'

## And Ollama also gives an OpenAI compatible endpoint

...and it's on your local machine!

If the next cell doesn't print "Ollama is running" then please open a terminal and run `ollama serve`

In [ ]:
requests.get("http://localhost:11434").content

### Download llama3.2 from meta

Change this to llama3.2:1b if your computer is smaller.

Don't use llama3.3 or llama4! They are too big for your computer..

In [ ]:
!ollama pull llama3.2

In [ ]:
OLLAMA_BASE_URL = "http://localhost:11434/v1"

ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key='ollama')

In [ ]:
# Get a fun fact

response = ollama.chat.completions.create(model="llama3.2", messages=[{"role": "user", "content": "Tell me a fun fact"}])

response.choices[0].message.content